# FM for materials

## Xavier Bresson 2025


In [1]:
# Libraries  
import time
import torch
import torch.nn as nn
import torch.optim as optim
import os
import matplotlib.pyplot as plt
import pickle
import numpy as np
import datetime  

from IPython.display import display, HTML # material visulazition
# import py3Dmol # !pip install py3Dmol   

# PyTorch version and GPU
print(torch.__version__)
if torch.cuda.is_available():
  print(torch.cuda.get_device_name(0))
  device = torch.device("cuda:2") # use GPU
else:
  device = torch.device("cpu")
print(device)


2.2.1+cu118
NVIDIA L40S
cuda:2


## Load datasets

In [2]:
# class of atom dictionary
class Dictionary:
    def __init__(self):
        self.type2idx = {}
        self.idx2type = []
        self.type2num_occurence = {}
        self.idx2num_occurence = []
    def show(self):
        for idx, atom_type in enumerate(self.idx2type):
            print(idx,'\t', atom_type,'\t number of occurences = {}'.format(self.idx2num_occurence[idx]))
    def __len__(self):
        return len(self.idx2type)
        
class Material:
    def __init__(self, dataset, lattices_matrix, atomic_numbers, frac_coords, structure,
                 material_id=None, formula=None, formula_reduced=None, lattice_lengths=None, lattice_angles=None, 
                 lattices_vectors=None, num_atoms=None, atoms_type=None, pair_dist=None, cif=None):
        self.dataset = dataset
        self.material_id = material_id
        self.formula = formula
        self.formula_reduced = formula_reduced
        self.num_atoms = num_atoms
        self.atomic_numbers = atomic_numbers
        self.atoms_type = atoms_type
        self.lattice_lengths = lattice_lengths
        self.lattice_angles = lattice_angles
        self.lattice_vectors = lattice_vectors
        self.lattice_matrix = lattice_matrix
        self.frac_coords = frac_coords
        self.pair_dist = pair_dist
        self.cif = cif
        self.structure = structure


In [3]:
print('Loading datasets..')
start = time.time()

# data_folder_pytorch = 'datasets/02_mp20/pytorch_v25/'
data_folder_pytorch = 'pytorch_v25/'
print(data_folder_pytorch)

with open(data_folder_pytorch + "atom_dict.pkl", "rb") as f:
    atom_dict = pickle.load(f)
with open(data_folder_pytorch + "train_dataset_pytorch.pkl", "rb") as f:
    dataset = pickle.load(f)
with open(data_folder_pytorch + "val_dataset_pytorch.pkl", "rb") as f:
    val_dataset = pickle.load(f)
print('Time:',time.time()-start)

print('num train materials:', len(dataset)) # 27136
print('num val materials:', len(val_dataset)) # 9047
# print('atom_dict.list_atoms :',atom_dict.list_atoms)
print('atom_dict.idx2type :',atom_dict.idx2type)
print('atom_dict.type2idx :',atom_dict.type2idx)
# print('atom_dict.type2atoNum :',atom_dict.type2atoNum)
# print('atom_dict.idx2atoNum_dic :',atom_dict.idx2atoNum_dic)
# print('atom_dict.idx2atoNum_vec :',atom_dict.idx2atoNum_vec)
print('atom_dict.type2num_occurence :',atom_dict.type2num_occurence)
print('num_atom_type :', len(atom_dict.idx2type))
idx2atoNum = atom_dict.idx2atoNum_vec

num_atom_type = len(atom_dict.idx2type)
print('num_atom_type :', num_atom_type)

idx = 45
print('')
for attr in dir(dataset[idx]):
    value = getattr(dataset[idx], attr)
    if not attr.startswith('__') and not attr.startswith('cif'):
        print(f"{attr}: {value}")

print('atom_dict.lattice_mean :', atom_dict.lattice_mean)
print('atom_dict.lattice_std :', atom_dict.lattice_std)
print('atom_dict.frac_coord_mean :', atom_dict.frac_coord_mean)
print('atom_dict.frac_coord_std :', atom_dict.frac_coord_std)



Loading datasets..
pytorch_v25/
Time: 17.871994495391846
num train materials: 27136
num val materials: 9047
atom_dict.idx2type : {0: 'Na', 1: 'Mn', 2: 'Co', 3: 'Ni', 4: 'O', 5: 'Nd', 6: 'Al', 7: 'Cu', 8: 'Li', 9: 'Ir', 10: 'C', 11: 'S', 12: 'N', 13: 'La', 14: 'Eu', 15: 'Yb', 16: 'Ga', 17: 'Pt', 18: 'Bi', 19: 'In', 20: 'Tb', 21: 'Cd', 22: 'Te', 23: 'As', 24: 'Ti', 25: 'Cs', 26: 'Mo', 27: 'I', 28: 'Si', 29: 'Ge', 30: 'Au', 31: 'K', 32: 'Sb', 33: 'Hg', 34: 'Y', 35: 'Lu', 36: 'B', 37: 'Fe', 38: 'Mg', 39: 'Sc', 40: 'F', 41: 'Sn', 42: 'Sm', 43: 'Tl', 44: 'Ca', 45: 'Cr', 46: 'Se', 47: 'Dy', 48: 'Ta', 49: 'Ru', 50: 'Hf', 51: 'Be', 52: 'Er', 53: 'H', 54: 'Zn', 55: 'Pd', 56: 'Sr', 57: 'Ba', 58: 'Pr', 59: 'Rh', 60: 'V', 61: 'Br', 62: 'Ce', 63: 'Cl', 64: 'Os', 65: 'Ag', 66: 'Rb', 67: 'P', 68: 'Tm', 69: 'Zr', 70: 'Re', 71: 'W', 72: 'Pa', 73: 'Pb', 74: 'Pm', 75: 'Ho', 76: 'U', 77: 'Ac', 78: 'Nb', 79: 'He', 80: 'Th', 81: 'Gd', 82: 'Tc', 83: 'Xe', 84: 'Np', 85: 'Pu', 86: 'Kr', 87: 'Ne', 88: 'Ar'}
atom

## Print dataset statistics

In [4]:
def group_molecules_per_size(dataset):
    mydict = {}
    for mat in dataset:
        num_atoms = mat.num_atoms.item()
        if num_atoms not in mydict:
            mydict[num_atoms] = []
        mydict[num_atoms].append(mat)
    return mydict

print('Train distribution')
group_dataset = group_molecules_per_size(dataset)
# what is the biggest molecule in the train set
max_mat_size = max(list( group_dataset.keys()))
print('Max num atoms : ', max_mat_size)
print('Number of train materials : ', len(dataset))
# distribution w.r.t. molecule size
print('\nDataset')
list_num_atoms = []
list_num_mats = []
data = group_dataset
for num_atom in range(max_mat_size+1):
    try: 
        print('number of materials of size {}: \t {}'.format(num_atom, len(data[num_atom])))
        list_num_atoms.append(num_atom)
        list_num_mats.append(len(group_dataset[num_atom]))
    except:
        pass
# remove materials with one atom
list_num_atoms = list_num_atoms[1:]
list_num_mats = list_num_mats[1:]
num_atoms = torch.tensor(list_num_atoms).long()
print('num_atoms:', num_atoms)
prob_mat_size = 100 * torch.tensor(list_num_mats).float() / torch.tensor(list_num_mats).float().sum() 
num_mat_per_size = torch.ceil(prob_mat_size).long()
print('num_mat_per_size:', num_mat_per_size, num_mat_per_size.sum())
# num_sym_group_class = 229
# print('num_sym_group_class:', num_sym_group_class)


print('\nValidation distribution')
group_val_dataset  = group_molecules_per_size(val_dataset)
# max_mat_size = max(list( group_val_dataset.keys()))
print('Max num atoms : ', max_mat_size)
print('Number of val materials : ', len(val_dataset))
list_num_atoms = []
list_num_mats = []
data = group_val_dataset
for num_atom in range(max_mat_size+1):
    try: 
        print('number of materials of size {}: \t {}'.format(num_atom, len(data[num_atom])))
        list_num_atoms.append(num_atom)
        list_num_mats.append(len(group_val_dataset[num_atom]))
    except:
        pass
# remove materials with one atom
list_num_atoms = list_num_atoms[1:]
list_num_mats = list_num_mats[1:]
num_atoms = torch.tensor(list_num_atoms).long()
print('num_atoms:', num_atoms)
prob_mat_size_val = 100 * torch.tensor(list_num_mats).float() / torch.tensor(list_num_mats).float().sum() 
num_mat_per_size_val = torch.ceil(prob_mat_size_val).long()
print('num_mat_per_size_val:', num_mat_per_size_val, num_mat_per_size_val.sum())



Train distribution
Max num atoms :  20
Number of train materials :  27136

Dataset
number of materials of size 1: 	 59
number of materials of size 2: 	 572
number of materials of size 3: 	 538
number of materials of size 4: 	 4144
number of materials of size 5: 	 1279
number of materials of size 6: 	 2297
number of materials of size 7: 	 572
number of materials of size 8: 	 2119
number of materials of size 9: 	 932
number of materials of size 10: 	 2640
number of materials of size 11: 	 361
number of materials of size 12: 	 2624
number of materials of size 13: 	 585
number of materials of size 14: 	 1770
number of materials of size 15: 	 390
number of materials of size 16: 	 1819
number of materials of size 17: 	 264
number of materials of size 18: 	 1443
number of materials of size 19: 	 287
number of materials of size 20: 	 2441
num_atoms: tensor([ 2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,
        20])
num_mat_per_size: tensor([ 3,  2, 16,  5,  9,  3,  8,

## Generate batch of pytorch materials of same size

In [5]:
# Generate batch of pytorch materials of same size
# A class to help drawing batches of materials having the same size
class MaterialSampler:
    def __init__(self, organized_dataset, bs , shuffle=True):  
        self.bs = bs
        self.num_mat =  { sz: len(list_of_mat)  for sz , list_of_mat in organized_dataset.items() }
        self.counter = { sz: 0   for sz in organized_dataset }
        if shuffle:
            self.order = { sz: np.random.permutation(num)  for sz , num in self.num_mat.items() }
        else:
            self.order = { sz: np.arange(num)  for sz , num in self.num_mat.items() } 
    def compute_num_batches_remaining(self):
        # return {sz:  ( self.num_mat[sz] - self.counter[sz] ) // self.bs  for sz in self.num_mat} 
        return {sz:  int(np.ceil((self.num_mat[sz] - self.counter[sz])/self.bs))  for sz in self.num_mat}
    def choose_material_size(self):
        num_batches = self.compute_num_batches_remaining()
        possible_sizes =  np.array( list( num_batches.keys()) )
        prob           =  np.array( list( num_batches.values() )   ) 
        prob =  prob / prob.sum()
        sz   = np.random.choice(  possible_sizes , p=prob )
        return sz
    def is_empty(self):
        num_batches = self.compute_num_batches_remaining()
        return sum( num_batches.values() ) == 0
    def draw_batch_of_materials(self,sz):  
        indices = self.order[sz][ self.counter[sz] : self.counter[sz] + self.bs]
        self.counter[sz] += self.bs  
        return indices


bs = 50
sampler = MaterialSampler(group_dataset, bs)

# get a batch
sz = sampler.choose_material_size()
print(sz)
indices = sampler.draw_batch_of_materials(sz) 
print(indices)
minibatch_node = torch.stack( [ group_dataset[sz][i].frac_coords for i in indices] )
print(minibatch_node.size())


7
[332 361 297  64  74 420 307 467 356 181 303 560  98  76 173 353  59 125
 192  29 486   1 234 121 349 246  23 412 278 216 175 422 280 475  60 310
 244 245 493 358 279 108 270 439 320 444 238 259 500 224]
torch.Size([50, 7, 3])


## Flow Matching model 

In [6]:
# Global constants
d = 128; num_heads = 4; num_layers = 6 # debug
d_head = 128; num_heads = 8; d = d_head*num_heads; num_layers = 12  # 77M 
d_head = 128; num_heads = 8; d = d_head*num_heads; num_layers = 4  # 26M 
d_head = 64; num_heads = 8; d = d_head*num_heads; num_layers = 4 # 7M debug
# d_head = 64; num_heads = 8; d = d_head*num_heads; num_layers = 8  # 13M 
# # d_head = 64; num_heads = 8; d = d_head*num_heads; num_layers = 16  # 25M 
# d_head = 128; num_heads = 8; d = d_head*num_heads; num_layers = 16  # 102M 
# # d_head = 92; num_heads = 12; d = d_head*num_heads; num_layers = 12  # llama-90M 
d_head = 64; num_heads = 12; d = d_head*num_heads; num_layers = 12  # llama-44M <==
drop = 0.05; drop_emb = drop; drop_att = drop; drop_mha = drop; drop_mlp = drop; drop_lax = drop;
dPE = d
print('d, num_layers, num_heads, drop_emb, drop_att, drop_mha, drop_mlp : ', d, num_layers, num_heads, drop_emb, drop_att, drop_mha, drop_mlp)

# # DM
# beta_1 = 0.0001
# beta_T = 0.02
# num_t = 1000
# print('beta_1, beta_T, num_t : ', beta_1, beta_T, num_t)

# RotPE
T_max = max_mat_size + 3 # max number of atoms + 3 
print('T_max : ', T_max)

bs = 50 # batch size
num_mat = len(dataset)

# EMA : exponential moving average
mu_ema = 0.999
print('mu_ema : ', mu_ema)


d, num_layers, num_heads, drop_emb, drop_att, drop_mha, drop_mlp :  768 12 12 0.05 0.05 0.05 0.05
T_max :  23
mu_ema :  0.999


In [ ]:
# Design Transformer architecture based on Llama2's architecture

# time steps generated by a skewed normal distribution towards t=1 
def skewed_timestep_sample(num_samples):
    P_mean = -1.2
    P_std = 1.2
    rnd_normal = torch.randn((num_samples,))
    sigma = (rnd_normal * P_std + P_mean).exp()
    time = 1 / (1 + sigma)
    time = torch.clip(time, min=0.0001, max=1.0)
    return time
# t = skewed_timestep_sample(num_samples=100000, device = device )
# plt.hist(t.cpu(), bins=100, edgecolor='black')
# plt.show()

def scalar_embedding(scalars, dim, scalar_weight):
    """
    INPUT t: Float tensor of shape (B,) 
          dim: dimension for the embeddings
    OUTPUT embedings: Float Tensor of shape (B,dim)
    Remark: the timesteps t in [0,1] are assumed to be in the range [0,1000] 
    Each scalar t receive the embedding 
    [ cos(omega_0 t) , ...., cos(omega_K t) , sin(omega_0 t), ... , sin(omega_K t) ]
    where K = dim//2
    and the freq omega_0 to omega_K range from omega_0 = 1 to omega_K = 1/10,000. 
    """
    scalars = scalar_weight * scalars # the timesteps t in [0,1] are re-scaled to be in the range [0,1000]
    assert dim % 2 == 0
    K = dim//2
    k_range = torch.arange(0,K, dtype=torch.float32) # shape K
    #omegas = 1.0 / ( 10000**(k_range/K)  )   # shape K
    # log_omega = -math.log(10000) * k_range/K # shape K
    log_omega = -torch.log(torch.tensor(10000.0)) * k_range/K # shape K
    omega = torch.exp(log_omega) # shape K
    omega = omega.to(device=scalars.device)
    exponents = scalars.unsqueeze(dim=-1) * omega       # shape  (B,K)
    cosine = torch.cos(exponents)  # shape  (B,K)
    sine = torch.sin(exponents) # shape  (B,K)
    embeddings = torch.cat( [cosine,sine] , dim = -1) # shape  (B,2K)
    return embeddings

# For ODE generation
def get_time_discretization(nfes, rho=7):
    step_indices = torch.arange(nfes, dtype=torch.float64)
    sigma_min = 0.002
    sigma_max = 80.0
    sigma_vec = (
        sigma_max ** (1 / rho)
        + step_indices / (nfes - 1) * (sigma_min ** (1 / rho) - sigma_max ** (1 / rho))
    ) ** rho
    sigma_vec = torch.cat([sigma_vec, torch.zeros_like(sigma_vec[:1])])
    time_vec = (sigma_vec / (1 + sigma_vec)).squeeze()
    t_samples = 1.0 - torch.clip(time_vec, min=0.0, max=1.0)
    return t_samples

# RMS Normalization 
# Root Mean Square Layer Normalization, Zhang, Sennrich, https://arxiv.org/pdf/1910.07467
class RMSNorm(torch.nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d))
    def forward(self, x):
        x_float = x.float() # for stability, float32
        norm_x_float = x_float * torch.rsqrt(x_float.pow(2).mean(-1, keepdim=True) + self.eps)
        output = self.weight * norm_x_float.type_as(x) 
        return output

# RotPE from RoFormer: Enhanced Transformer with Rotary Position Embedding, Jianlin Su et-al, https://arxiv.org/pdf/2104.09864
# PE_{t,k} = exp(i omega_k t) where omega_k = (10000)^(-2k/d) where k = 0, 1, ..., (d/2)-1
# t = 0, 1, ... , T_max-1 where T_max is the maximal sequence length
# PE in C^{ T_max x d/2} where C is the complex set
def precompute_complex_PE(T_max, d): # T_max=3, d=4
    k = torch.arange(d//2).float() # size=[d/2=2], k = [0., 1.] 
    omegas = 10000 ** (-2*k/d )    # size=[d/2=2], [1.0000, 0.0100]
    t = torch.arange(T_max)        # size=[T_max=3], t = [0, 1, 2]
    angles = torch.outer(t,omegas).float() # outer product, shape=[T_max=3, d/2=2], [ [0.0000, 0.0000], [1.0000, 0.0100], [2.0000, 0.0200] ] 
    PE_cplx = torch.polar(torch.ones_like(angles), angles) # size=[T_max, dh/2], torch.polar(abs, angle) = abs⋅cos(angle)+abs⋅sin(angle)⋅j 
          # = [ [ 1.0000+0.0000j,  1.0000+0.0000j], [ 0.5403+0.8415j,  0.9999+0.0100j], [-0.4161+0.9093j,  0.9998+0.0200j] ]
    return PE_cplx

# Apply rotary embeddings to input tensors using a frequency tensor
def apply_rotary_emb(x, PE_cplx):
    bsz, n_head, T, d = x.size()              # size=[bsz, n_head, T, d], [50, 4, 8, 32]
    y = x.view(bsz, n_head, T, d//2, 2)       # size=[bsz, n_head, T, d/2, 2], [50, 4, 8, 16, 2]
    z = torch.view_as_complex(y.float())      # size=[bsz, n_head, T, d/2] complex tensor, [50, 4, 8, 16] 
    z = z * PE_cplx[:T,:].view(1, 1, T, d//2) # size=[bsz, n_head, T, d/2], original size=[T, d/2], [50, 4, 8, 16]
    y = torch.view_as_real(z)                 # size=[bsz, n_head, T, d/2, 2] real tensor, [50, 4, 8, 16, 2]
    y = y.view(bsz, n_head, T, d).type_as(x)  # size=[bsz, n_head, T, d], [50, 4, 8, 32]
    return y

# Multi-Head Attention
class MHA(nn.Module):
    def __init__(self, d, n_heads):  
        super().__init__()
        d_head = d // n_heads # TODO check dims <==
        self.d_head = d_head
        self.n_heads = n_heads
        self.Q = nn.Linear(d_head * n_heads, d, bias=False) # d = d_head * n_heads
        self.K = nn.Linear(d_head * n_heads, d, bias=False) # d = d_head * n_heads
        self.V = nn.Linear(d_head * n_heads, d, bias=False) # d = d_head * n_heads
        self.WO = nn.Linear(              d, d, bias=True)
        self.drop_att = nn.Dropout(drop_att) 
    def forward(self, x, PE_cplx):
        q, k, v = self.Q(x), self.K(x), self.V(x)           # [bs, n+3, d] = [50, 8, 128]
        bsz = x.size(0); setsize = x.size(1)
        q = q.view(bsz, setsize, self.n_heads, self.d_head) # [bs, n+3, n_heads, d_head] = [50, 8, 4, 32]
        k = k.view(bsz, setsize, self.n_heads, self.d_head) # [bs, n+3, n_heads, d_head] = [50, 8, 4, 32]
        v = v.view(bsz, setsize, self.n_heads, self.d_head) # [bs, n+3, n_heads, d_head] = [50, 8, 4, 32]
        q = q.transpose(1, 2)                               # [bs, n_heads, n+3, d_head] = [50, 4, 8, 32]
        k = k.transpose(1, 2)                               # [bs, n_heads, n+3, d_head] = [50, 4, 8, 32]
        v = v.transpose(1, 2)                               # [bs, n_heads, n+3, d_head] = [50, 4, 8, 32]
        q = apply_rotary_emb(q, PE_cplx)                    # [bs, n_heads, n+3, d_head] = [50, 4, 8, 32] 
        k = apply_rotary_emb(k, PE_cplx)                    # [bs, n_heads, n+3, d_head] = [50, 4, 8, 32] 
        att_scores = ( q @ k.transpose(2, 3) ) / torch.sqrt(torch.tensor(self.d_head)) # [bs, n_heads, n+3, n+3] = [50, 4, 8, 8]
        att_scores = torch.softmax(att_scores.float(), dim=3).type_as(x) # softmax_rows => sum_rows att_scores = 1
        att_scores = self.drop_att(att_scores) 
        h = att_scores @ v                                  # [bs, n_heads, n+3, d_head] = [50, 4, 8, 32]
        h = h.transpose(1, 2).contiguous().view(bsz, setsize, -1) # [bs, n+3, d] = [50, 8, 128]
        h = self.WO(h)                                            # [bs, n+3, d] = [50, 8, 128] 
        return h

# SwiGLU: GLU Variants Improve Transformer, Noam Shazeer, https://arxiv.org/pdf/2002.05202
class MLP_SwiGLU(nn.Module):
    def __init__(self, d):  
        super().__init__()
        hidden_d = int( 2/3 * d ) 
        self.linear1 = nn.Linear(d, hidden_d, bias=False)
        self.linear2 = nn.Linear(hidden_d, d, bias=False)
        self.linear3 = nn.Linear(d, hidden_d, bias=False)
    def forward(self, x):
        h = self.linear2(nn.SiLU()(self.linear1(x)) * self.linear3(x)) # [bs, n+3, d] = [50, 8, 128]
        return h

# Block Transformer
class BlockGT(nn.Module):
    def __init__(self, d, num_heads):  
        super().__init__()
        self.RMSNorm_MHA = RMSNorm(d)
        self.MHA = MHA(d, num_heads)
        self.drop_mha = nn.Dropout(drop_mha) 
        self.RMSNorm_MLP = RMSNorm(d)
        self.MLP_SwiGLU = MLP_SwiGLU(d)
        self.drop_mlp = nn.Dropout(drop_mlp) 
    def forward(self, x, PE_cplx):
        x_norm = self.RMSNorm_MHA(x)      # [bs, n+3, d] = [50, 8, 128]
        x = x + self.drop_mha(self.MHA(x_norm, PE_cplx)) # [bs, n+3, d] = [50, 8, 128]
        x_norm = self.RMSNorm_MLP(x)      # [bs, n+3, d] = [50, 8, 128]
        x = x + self.drop_mlp(self.MLP_SwiGLU(x_norm))   # [bs, n+3, d] = [50, 8, 128]
        return x 

class Transformer(nn.Module):
    def __init__(self):
        super().__init__()
        # MLP Best Practice : LL -> Activation -> LN -> Drop
        hidden_d = int( 3/2 * d ) 
        # Encoder 
        # self.t_enc = nn.Sequential( nn.Linear(d, hidden_d), nn.SiLU(), RMSNorm(hidden_d), nn.Dropout(drop), nn.Linear(hidden_d, d) )
        self.t_enc = nn.Sequential( nn.Linear(d, hidden_d), nn.SiLU(), nn.Dropout(drop), nn.Linear(hidden_d, d) )
        self.x_t_enc = nn.Sequential( nn.Linear(3, hidden_d), nn.SiLU(), nn.Dropout(drop), nn.Linear(hidden_d, d) )
        self.a_t_enc = nn.Sequential( nn.Linear(num_atom_type, hidden_d), nn.SiLU(), nn.Dropout(drop), nn.Linear(hidden_d, d) )
        self.l_t_enc = nn.Sequential( nn.Linear(9, hidden_d), nn.SiLU(), nn.Dropout(drop), nn.Linear(hidden_d, d) )
        # PE
        self.PE_cplx = precompute_complex_PE(T_max, d//num_heads) 
        # Transformer layers
        self.gt_layers = nn.ModuleList( [BlockGT(d, num_heads) for _ in range(num_layers)] )
        # Decoder
        # self.ut_x_dec = nn.Sequential( RMSNorm(d), nn.Linear(d, hidden_d), nn.SiLU(), nn.Dropout(drop), nn.Linear(hidden_d, 3) )
        self.ut_x_dec = nn.Sequential( nn.Linear(d, hidden_d), nn.SiLU(), nn.Dropout(drop), nn.Linear(hidden_d, 3) )
        self.ut_a_dec = nn.Sequential( nn.Linear(d, hidden_d), nn.SiLU(), nn.Dropout(drop), nn.Linear(hidden_d, num_atom_type) )
        self.ut_l_dec = nn.Sequential( nn.Linear(d, hidden_d), nn.SiLU(), nn.Dropout(drop), nn.Linear(hidden_d, 9) )
        # with torch.no_grad():
        #     self.ut_x_dec[-1].weight.zero_(); self.ut_x_dec[-1].bias.zero_()
        #     self.ut_a_dec[-1].weight.zero_(); self.ut_a_dec[-1].bias.zero_()
        #     self.ut_l_dec[-1].weight.zero_(); self.ut_l_dec[-1].bias.zero_()
    def forward(self, x_t, a_t, l_t, sample_t): # [bs, n, 3], [bs, n], [bs, 3, 3] 
        p_t = self.t_enc( scalar_embedding(sample_t, d, 1000.0) ) # [bs, d] 
        # print('p_t:', p_t.size())     
        h_x_t = self.x_t_enc(x_t) # [bs, n, d] 
        # print('h_x_t:', h_x_t.size())
        h_a_t =  self.a_t_enc(a_t) # [bs, n, d]
        # print('h_a_t:', h_a_t.size())    
        h_l_t = self.l_t_enc(l_t.reshape(-1, 9) ).unsqueeze(1) # [bs, 1, d] 
        # print('h_l_t:', h_l_t.size())
        # combine [Lattice_CLS, Atom0, Atom1,...]
        h_atoms = h_x_t + h_a_t # [bs, n, d]
        # print('h_atoms:', h_atoms.size())
        h_t = torch.cat([h_l_t, h_atoms], dim=1) # [bs, n+1, d]
        # h_t = h_x_t + h_a_t + h_l_t # [bs, n, d] 
        # print('h_t:', h_t.size())
        for gt_layer in self.gt_layers:
            h_t = h_t + p_t.unsqueeze(1)
            h_t = gt_layer(h_t, self.PE_cplx.to(h_t.device))  # [bs, n, d] 
        # print('h_t:', h_t.size())
        # split back: lattice (index 0) vs atoms (indices 1..N)
        h_t_l = h_t[:, 0, :] # [bs, d] 
        # print('h_t_l:', h_t_l.size())
        h_t_atoms = h_t[:, 1:, :] # [bs, n, d] 
        # print('h_t_atoms:', h_t_atoms.size())
        ut_x_pred = self.ut_x_dec(h_t_atoms)  # [bs, n, 3] 
        # print('ut_x_pred:', ut_x_pred.size())
        ut_a_pred = self.ut_a_dec(h_t_atoms)  # [bs, n, num_atom_type] 
        # print('ut_a_pred:', ut_a_pred.size())
        ut_l_pred = self.ut_l_dec(h_t_l).reshape(-1,3,3) # [bs, 3, 3] 
        # print('ut_l_pred:', ut_l_pred.size())
        return ut_x_pred, ut_a_pred, ut_l_pred
        
class FM(nn.Module):

    def __init__(self):
        super().__init__()
        self.Transformer = Transformer()
    
    def forward(self, x_t, a_t, l_t, sample_t): # predict velocity
        u_t_x_pred, u_t_a_pred, u_t_l_pred = self.Transformer(x_t, a_t, l_t, sample_t)
        return u_t_x_pred, u_t_a_pred, u_t_l_pred
    
    def improved_euler(self, x0, a0, l0, time_grid): 
        bs = x0.shape[0]
        x = x0 # [bs, n, 3]
        a = a0 # [bs, n]
        l = l0 # [bs, 3, 3]
        for i in range( len(time_grid)-1 ):
            t = time_grid[i].item()
            t_prime = time_grid[i+1].item()
            dt = t_prime - t
            t1 = torch.full(size=(bs,) , fill_value = t).to(device) # [bs]
            with torch.no_grad():
                v1_x, v1_a, v1_l = self.forward(x, a, l, t1) 
            x_temp = x + dt * v1_x # [bs, n, 3]
            a_temp = a + dt * v1_a # [bs, n]
            l_temp = l + dt * v1_l # [bs, 3, 3]
            t2 = torch.full(size=(bs,) , fill_value = t_prime).to(device)
            with torch.no_grad():
                v2_x, v2_a, v2_l = self.forward(x_temp, a_temp, l_temp, t2)
            x = x + dt * (v1_x + v2_x)/2 # [bs, n, 3]
            a = a + dt * (v1_a + v2_a)/2 # [bs, n]
            l = l + dt * (v1_l + v2_l)/2 # [bs, 3, 3]
        return x, a, l

    def generate_process_fm(self, num_mol, size_mol, num_time_steps=100): # 
        time_grid = get_time_discretization(num_time_steps)
        X_0 = torch.randn(size=(num_mol, size_mol, 3), dtype=torch.float32).to(device)
        A_0 = torch.randn(size=(num_mol, size_mol, num_atom_type), dtype=torch.float32).to(device)
        L_0 = torch.randn(size=(num_mol, 3, 3), dtype=torch.float32).to(device)
        X_1, A_1, L_1 = self.improved_euler(X_0, A_0, L_0, time_grid)
        return X_1, A_1, L_1

        
def count(modules):
    # Handles both single modules and lists of modules
    if isinstance(modules, list):
        return sum(sum(p.numel() for p in m.parameters() if p.requires_grad) for m in modules) / 1e6
    return sum(p.numel() for p in modules.parameters() if p.requires_grad) / 1e6


# Instantiate the network
net = FM()
net = net.to(device)
def display_num_param(net):
    nb_param = 0
    for param in net.parameters():
        nb_param += param.numel()
    return nb_param / 1e6
print('num_param (million):', display_num_param(net))

print("Block Time:", count([net.Transformer.t_enc]))
print("Block Encoders:", count([net.Transformer.x_t_enc, net.Transformer.a_t_enc, net.Transformer.l_t_enc]))
print("Block GT Layers:", count([net.Transformer.gt_layers]))
print("Block Decoders:", count([net.Transformer.ut_x_dec, net.Transformer.ut_a_dec, net.Transformer.ut_l_dec]))

# Exponential Moving Average (EMA)
ema_net = FM()
ema_net = ema_net.to(device)
ema_net.load_state_dict(net.state_dict())

# Test the forward pass, backward pass and gradient update with a single batch
init_lr = 0.001
optimizer = torch.optim.AdamW(net.parameters(), lr=init_lr)
# scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.95, patience=1, verbose=True)

# Input
sampler = MaterialSampler(group_dataset, bs)
# print('sampler.num_mat :', sampler.num_mat)
num_batches_remaining = sampler.compute_num_batches_remaining()
# print('num_batches_remaining :', num_batches_remaining)
sz = sampler.choose_material_size()
# print('sz :',sz)
indices = sampler.draw_batch_of_materials(sz) 
# print('indices :', len(indices), indices)
batch_size = len(indices)
batch_sample_t = skewed_timestep_sample(batch_size).to(device) # (bs,1,1) random value in [0,1]
# print('batch_sample_t', batch_sample_t.size()) # [50, 1, 1]
X1 = torch.stack( [ group_dataset[sz][i].frac_coords for i in indices] ).float().to(device) # [bs, n, 3] 
X1 = X1 + torch.empty(batch_size,3).uniform_(0.0, 1.0).to(device).unsqueeze(1); X1 = X1 - X1.floor() # data augmentation: apply translation to frac coordinates
def atom_reordering(frac_coords):
    values = 100.0 * frac_coords[:,:,0] + 10.0 * frac_coords[:,:,1] + 1.0 * frac_coords[:,:,2]
    new_index_order = torch.argsort(values)
    return new_index_order
new_index_order = atom_reordering(X1)
X1 = torch.gather(X1, 1, new_index_order.unsqueeze(-1).expand_as(X1)) # [bs, n, 3] 
X1 = ( X1 - atom_dict.frac_coord_mean ) / atom_dict.frac_coord_std
X0 = torch.randn_like(X1).to(device) # [bs, n, 3]    
Xt = ( 1 - batch_sample_t.view(batch_size,1,1) ) * X0 + batch_sample_t.view(batch_size,1,1) * X1 # [bs, n, 3] 
Ut_X = X1 - X0 # [bs, n, 3] 
# print('Ut_X :', Ut_X.size())
A1 = torch.stack( [ group_dataset[sz][i].atoms_type for i in indices] ).long().to(device) # [bs, n] 
A1 = torch.gather(A1, 1, new_index_order) # [bs, n]
A1 = torch.nn.functional.one_hot(A1, num_atom_type) # [bs, n, n_atom_type] = [50, 12, 89]
A1 = 2.0 * ( A1 - 0.5 )
A0 = torch.randn_like(A1).to(device) # [bs, n] 
At = ( 1 - batch_sample_t.view(batch_size,1,1) ) * A0 + batch_sample_t.view(batch_size,1,1) * A1 # [bs, n] 
Ut_A = A1 - A0 # [bs, n] 
# print('Ut_A :', Ut_A.size())
L1 = torch.stack( [ group_dataset[sz][i].lattice_matrix for i in indices] ).float().to(device) # [bs, 3, 3] 
Q, R = torch.linalg.qr(torch.randn(batch_size, 3, 3, device=device)) # QR Decomposition, Q rotation matrix
L1 = torch.bmm(L1, Q * torch.linalg.det(Q).sign().view(-1, 1, 1)) # Enforce Determinant = +1 (Pure Rotation)
L1 = ( L1 - atom_dict.lattice_mean ) / atom_dict.lattice_std # [bs, 3, 3] 
L0 = torch.randn_like(L1).to(device) # [bs, 3, 2] 
Lt = ( 1 - batch_sample_t.view(batch_size,1,1) ) * L0 + batch_sample_t.view(batch_size,1,1) * L1 # [bs, 3, 3] 
Ut_L = L1 - L0 # [bs, 3, 3] 
# print('Ut_L :', Ut_L.size())
# cl_gt = torch.stack( [ group_dataset[sz][i].sym_group_number for i in indices] ).long().to(device) - 1 # [bs]
# print('cl_gt :', cl_gt.size())

# Forward pass
Ut_X_pred, Ut_A_pred, Ut_L_pred = net(Xt, At, Lt, batch_sample_t) # [bs, n+3, n_atom_type+4]=[50, 8, 60], [50, 229]
print('Ut_X_pred :', Ut_X_pred.size())
print('Ut_A_pred :', Ut_A_pred.size())
print('Ut_L_pred :', Ut_L_pred.size())

# Compute loss, backward pass, GD
mse_loss = torch.nn.MSELoss()
loss_FM = mse_loss(Ut_X_pred, Ut_X) + mse_loss(Ut_A_pred, Ut_A) + mse_loss(Ut_L_pred, Ut_L) 
loss = loss_FM
# lambda_t = 10 * (1.0 - batch_sample_t.squeeze())**3 # [bs]
# # print('lambda_t :', lambda_t.size(), lambda_t)
# loss_groups = nn.CrossEntropyLoss(reduce='none')(cl_pred, cl_gt).mean()
# # print('loss_groups :', loss_groups.size())
# loss = loss_FM + loss_groups
optimizer.zero_grad()
loss.backward()
torch.nn.utils.clip_grad_norm_(net.parameters(), 0.25) # grad_norm_clip=1.0
optimizer.step()

# # Generate a few samples
# with torch.no_grad():
#     batch_x1, batch_a1, batch_l1 = net.generate_process_fm(4, 7, 100)
#     print('batch_x1',batch_x1.size())
#     print('batch_a1',batch_a1.size())
#     print('batch_l1',batch_l1.size())



num_param (million): 49.816805
Block Time: 1.771392
Block Encoders: 2.77632
Block GT Layers: 42.494976
Block Decoders: 2.774117


In [ ]:
## Training loop
try:
    del net, ema_net
except NameError:
    pass
net = FM()
net = net.to(device)

# Exponential Moving Average (EMA)
ema_net = FM()
ema_net = ema_net.to(device)
ema_net.load_state_dict(net.state_dict())
def update_ema_net(mu_ema, num_batches): 
    nu_ema = min( mu_ema , (1 + num_batches) / (10 + num_batches) )
    ema_current = ema_net.state_dict()
    for name, param in net.named_parameters():
        if param.requires_grad:
            ema_current[name].data = (1 - nu_ema) * param.data + nu_ema * ema_current[name].data
    ema_net.load_state_dict(ema_current) 


# Optimizer
# schedule
max_lr = 0.0003; min_lr = 0.00003; nb_epochs = 3000; bs = 128 # 
max_lr = 0.0003; min_lr = 0.00003; nb_epochs = 3000; bs = 512 # 
max_lr = 0.0003; min_lr = 0.00001; nb_epochs = 10000; bs = 512 # 
max_lr = 0.0003; min_lr = 0.0000001; nb_epochs = 100000; bs = 512 # 1e-7
print(f"nb_epochs= {nb_epochs}, bs= {bs}, max_lr= {max_lr:.6f}, min_lr= {min_lr:.6f}")
# schedule
num_warm_up_iterations = 5000
warmup_steps = num_warm_up_iterations
print("number of warmup steps =", warmup_steps)
num_step_per_epoch = num_mat // bs
print("number of step per epoch =", num_step_per_epoch)
num_warm_up_epochs = warmup_steps // num_step_per_epoch
print("number of warmup epochs =", num_warm_up_epochs)
max_steps = nb_epochs * num_step_per_epoch
print("max number of processed batches =", max_steps)
def get_lr(it):
    # 1) linear warmup for warmup_iters steps
    if it < warmup_steps:
        return max_lr * (it+1) / warmup_steps
    # 2) if it > lr_decay_iters, return min learning rate
    if it > max_steps:
        return min_lr
    # 3) in between, use cosine decay down to min learning rate
    decay_ratio = (it - warmup_steps) / (max_steps - warmup_steps)
    assert 0 <= decay_ratio <= 1 
    import math
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio)) # coeff starts at 1 and goes to 0
    return min_lr + coeff * (max_lr - min_lr)

import inspect
def configure_optimizers(net, weight_decay, learning_rate, device_type):
    # start with all of the candidate parameters (that require grad)
    param_dict = {pn: p for pn, p in net.named_parameters()}
    param_dict = {pn: p for pn, p in param_dict.items() if p.requires_grad}
    # create optim groups. Any parameters that is 2D will be weight decayed, otherwise no.
    # i.e. all weight tensors in matmuls + embeddings decay, all biases and layernorms don't.
    decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
    nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
    optim_groups = [ {'params': decay_params, 'weight_decay': weight_decay},
                     {'params': nodecay_params, 'weight_decay': 0.0} ]
    num_decay_params = sum(p.numel() for p in decay_params)
    num_nodecay_params = sum(p.numel() for p in nodecay_params)
    print(f"num decayed parameter tensors: {len(decay_params)}, with {num_decay_params:,} parameters")
    print(f"num non-decayed parameter tensors: {len(nodecay_params)}, with {num_nodecay_params:,} parameters")
    # Create AdamW optimizer and use the fused version if it is available
    fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
    # use_fused = fused_available and device_type == "cuda"
    use_fused = fused_available and torch.cuda.is_available()
    print(f"using fused AdamW: {use_fused}")
    optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=(0.9, 0.95), eps=1e-8, fused=use_fused)
    return optimizer
optimizer = configure_optimizers(net, weight_decay=0.1, learning_rate=max_lr, device_type=device)


# Saving checkpoint
time_stamp = datetime.datetime.now().strftime("%y-%m-%d--%H-%M-%S")
checkpoint_dir = os.path.join("checkpoint")
if not os.path.exists(checkpoint_dir):
    os.makedirs(checkpoint_dir)
epoch_ckpt = tot_time_ckpt = perc_valid_mol_ckpt = perc_valid_mol_best = perc_valid_mol_ema_best = num_warmup_batch = 0

file = checkpoint_file = checkpoint_dir + '/checkpoint_' + time_stamp + '.txt'
print(file)
file_object = open(file, 'w')
print('num_param (million):', display_num_param(net))
import subprocess
hostname = subprocess.run(['hostnamectl', '--pretty'], capture_output=True, text=True, check=True).stdout.strip()
file_object.write(f"hostname: {hostname}\n")
print('hostname:', hostname)
file_object.write(f"device: {device}\n")
print('device:', device)
file_object.write(f"file: {file}\n")
import ipynbname 
notebook_name = ipynbname.name()
file_object.write(f"notebook_name: {notebook_name}\n")
print('notebook_name:', notebook_name)
file_object.close()

import argparse
ddpm_args = argparse.ArgumentParser().parse_args([])
ddpm_args.d_head = d_head
ddpm_args.num_heads = num_heads
ddpm_args.d = d
ddpm_args.num_layers = num_layers
ddpm_args.drop = drop
ddpm_args.dPE = dPE
ddpm_args.T_max = T_max
ddpm_args.mu_ema = mu_ema
ddpm_args.num_mat = num_mat
ddpm_args.nb_epochs = nb_epochs
ddpm_args.bs = bs
ddpm_args.max_lr = max_lr
ddpm_args.min_lr = min_lr
ddpm_args.num_warm_up_epochs = num_warm_up_epochs
ddpm_args.num_param = display_num_param(net)
ddpm_args.time_stamp = time_stamp
print('ddpm args:', ddpm_args)
file_object = open(file, 'a+'); file_object.write(f"{ddpm_args}\n"); file_object.close()

def atom_reordering(frac_coords):
    values = 100.0 * frac_coords[:,:,0] + 10.0 * frac_coords[:,:,1] + 1.0 * frac_coords[:,:,2]
    new_index_order = torch.argsort(values)
    return new_index_order

from predict_vsu.predict_vsu import predict_vsu

list_loss = []; epoch_loss_plot = []; loss_plot = []; vsu_plot = [];
vsu = 0; best_vsu = 0;
start=time.time()
print('nb_epochs:', nb_epochs, '\n')
tot_num_batches = 0 # Optimizer v2
# for epoch in range(nb_epochs):
epoch = 0
while epoch <= 100000:

    running_loss = 0.0
    num_batches = 0
    net.train()

    sampler = MaterialSampler(group_dataset, bs)
    while(not sampler.is_empty()):

        # Forward pass
        num_batches_remaining = sampler.compute_num_batches_remaining()
        sz = sampler.choose_material_size()
        indices = sampler.draw_batch_of_materials(sz) 
        batch_size = len(indices)
        batch_sample_t = skewed_timestep_sample(batch_size).to(device) # (bs,1,1) random value in [0,1]
        X1 = torch.stack( [ group_dataset[sz][i].frac_coords for i in indices] ).float().to(device) # [bs, n, 3] 
        X1 = X1 + torch.empty(batch_size,3).uniform_(0.0, 1.0).to(device).unsqueeze(1); X1 = X1 - X1.floor() # data augmentation: apply translation to frac coordinates
        new_index_order = atom_reordering(X1) # re-order atoms
        X1 = torch.gather(X1, 1, new_index_order.unsqueeze(-1).expand_as(X1)) # [bs, n, 3] re-order atoms
        X1 = ( X1 - atom_dict.frac_coord_mean ) / atom_dict.frac_coord_std
        X0 = torch.randn_like(X1).to(device) # [bs, n, 3]    
        Xt = ( 1 - batch_sample_t.view(batch_size,1,1) ) * X0 + batch_sample_t.view(batch_size,1,1) * X1 # [bs, n, 3] 
        Ut_X = X1 - X0 # [bs, n, 3] 
        A1 = torch.stack( [ group_dataset[sz][i].atoms_type for i in indices] ).long().to(device) # [bs, n] 
        A1 = torch.gather(A1, 1, new_index_order) # [bs, n] re-order atoms
        A1 = torch.nn.functional.one_hot(A1, num_atom_type) # [bs, n, n_atom_type] = [50, 12, 89]
        A1 = 2.0 * ( A1 - 0.5 )
        A0 = torch.randn_like(A1).to(device) # [bs, n] 
        At = ( 1 - batch_sample_t.view(batch_size,1,1) ) * A0 + batch_sample_t.view(batch_size,1,1) * A1 # [bs, n] 
        Ut_A = A1 - A0 # [bs, n] 
        L1 = torch.stack( [ group_dataset[sz][i].lattice_matrix for i in indices] ).float().to(device) # [bs, 3, 3]
        # Q, R = torch.linalg.qr(torch.randn(batch_size, 3, 3, device=device)) # QR Decomposition, Q rotation matrix
        # L1 = torch.bmm(L1, Q * torch.linalg.det(Q).sign().view(-1, 1, 1)) # Enforce Determinant = +1 (Pure Rotation)
        L1 = ( L1 - atom_dict.lattice_mean ) / atom_dict.lattice_std # [bs, 3, 3] 
        L0 = torch.randn_like(L1).to(device) # [bs, 3, 2] 
        Lt = ( 1 - batch_sample_t.view(batch_size,1,1) ) * L0 + batch_sample_t.view(batch_size,1,1) * L1 # [bs, 3, 3] 
        Ut_L = L1 - L0 # [bs, 3, 3]  
        Ut_X_pred, Ut_A_pred, Ut_L_pred = net(Xt, At, Lt, batch_sample_t) # [bs, n+3, n_atom_type+4]=[50, 8, 60], [50, 229]

        # Compute loss, backward pass, GD
        mse_loss = torch.nn.MSELoss()
        loss_FM = ( mse_loss(Ut_X_pred, Ut_X) + mse_loss(Ut_A_pred, Ut_A) + mse_loss(Ut_L_pred, Ut_L) ) / 3.0
        loss = loss_FM
        optimizer.zero_grad()
        loss.backward()
        norm = torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0) 

        # Optimizer
        lr = get_lr(tot_num_batches)
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr
        optimizer.step()

        # EMA 
        update_ema_net(mu_ema, tot_num_batches)

        # COMPUTE STATS
        running_loss += loss.detach().item()
        num_batches += 1
        tot_num_batches += 1

        
    # store loss every x epochs and plot
    avg_epoch_loss = running_loss / num_batches
    freq_plot = 500
    # freq_plot = 2 # DEBUG
    if epoch>=1 and epoch%freq_plot==0:
        
        epoch_loss_plot.append(epoch)
        loss_plot.append(avg_epoch_loss)
        vsu, _ = predict_vsu(ema_net, atom_dict, device, 10000)      
        vsu_plot.append(vsu)
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 12))
        # --- Subplot 1: Loss ---
        idx_min_loss = loss_plot.index(min(loss_plot))
        epoch_min_loss = epoch_loss_plot[idx_min_loss]
        ax1.plot(epoch_loss_plot, loss_plot, label='Training Loss')
        ax1.set_xlabel('Epoch'); ax1.set_ylabel('Training Loss')
        ax1.set_title(f'Training Loss with minimum {min(loss_plot):.4f} and epoch {epoch_min_loss}')
        ax1.grid(True); ax1.legend()
        # --- Subplot 2: Vsu ---
        idx_max_vsu = vsu_plot.index(max(vsu_plot))
        epoch_max_vsu = epoch_loss_plot[idx_max_vsu]
        ax2.plot(epoch_loss_plot, vsu_plot, label='VSU')
        ax2.set_xlabel('Epoch'); ax2.set_ylabel('VSU')
        ax2.set_title(f'VSU with maximum {max(vsu_plot):.3f} and epoch {epoch_max_vsu}')
        ax2.grid(True); ax2.legend()
        # Save and Close
        plt.tight_layout() # Adjusts spacing to prevent overlap
        plt.savefig(checkpoint_dir + '/checkpoint_' + time_stamp + '_metrics_plot.png')
        plt.close()   
        
        # Saving best vsu checkpoint
        current_vsu = vsu 
        if current_vsu > best_vsu:
            # remove previous best model
            if best_vsu>0:
                file_to_remove = checkpoint_dir + "/checkpoint_best" + "_vsu_" + str(best_vsu)[:5] + "_epochs_" + str(best_vsu_epoch) + "_" + time_stamp + '.pkl'
                import os
                os.remove(file_to_remove)
            best_vsu = vsu
            best_vsu_epoch = epoch
            torch.save({
                'epoch': epoch,
                'best_vsu': best_vsu,
                'best_vsu_epoch': best_vsu_epoch,
                'ema_net': ema_net.state_dict(),
                'notebook_name': notebook_name,
                }, '{}.pkl'.format(checkpoint_dir + "/checkpoint_best" + "_vsu_" + str(best_vsu)[:5] + "_epochs_" + str(best_vsu_epoch) + "_" + time_stamp))

    # print stats 
    list_loss.append(avg_epoch_loss)
    moving_loss = torch.Tensor(list_loss)[-10:].mean().item() # mean lost value over the last 10 epochs
    elapsed = time.time()-start 
    
    if not epoch%1:
        line = 'epoch= ' + str(epoch) + '  t(min)= ' + str((elapsed)/60)[:6] + \
        '  t(day)= ' + str((elapsed)/3600/24)[:4] + '  lr= ' + \
        '{:.7f}'.format(optimizer.param_groups[0]['lr']) + '  norm= ' + str(norm.detach().item())[:4] + \
        '  loss_train= ' + str(avg_epoch_loss)[:6] + '  mov= ' + str(moving_loss)[:6]  + \
        '  vsu= ' + str(vsu)[:6] + '  best_vsu= ' + str(best_vsu)[:6] 
        print(line)
        file_object = open(file, 'a+'); file_object.write(f"{line}\n"); file_object.close()

    # Saving very 5000 epochs
    if not epoch%5000 and epoch>0:
        torch.save({
            'epoch': epoch,
            'tot_time': elapsed,
            'loss': avg_epoch_loss,
            'moving_loss': moving_loss,
            'ema_net': ema_net.state_dict(),
            'notebook_name': notebook_name,
            }, '{}.pkl'.format(checkpoint_dir + "/checkpoint_" + str(epoch) + "_epochs_" + time_stamp ))
    
    epoch += 1



nb_epochs= 100000, bs= 512, max_lr= 0.000300, min_lr= 0.000000
number of warmup steps = 5000
number of step per epoch = 53
number of warmup epochs = 94
max number of processed batches = 5300000
num decayed parameter tensors: 98, with 49,777,920 parameters
num non-decayed parameter tensors: 50, with 38,885 parameters
using fused AdamW: True
checkpoint/checkpoint_26-02-10--00-08-50.txt
num_param (million): 49.816805
hostname: deeplearn22
device: cuda:2
notebook_name: 125_fm_llama_mp20_vsun_xb_v1
ddpm args: Namespace(d_head=64, num_heads=12, d=768, num_layers=12, drop=0.05, dPE=768, T_max=23, mu_ema=0.999, num_mat=27136, nb_epochs=100000, bs=512, max_lr=0.0003, min_lr=1e-07, num_warm_up_epochs=94, num_param=49.816805, time_stamp='26-02-10--00-08-50')
nb_epochs: 100000 

